# gold_vuln_daily_build
`active_cve_set` + `silver_epss` + `silver_kev` + `silver_vuln_alias` + `silver_epss_distribution` → `gold_vuln_daily`.
Grain: `cve_id` × `date_key`. "어제보다 올랐다"를 계산할 수 있는 유일한 테이블 — 여기가 있어야 30일 그래프도, EPSS_SPIKE 판정도 가능하다.

실행 모드:
- **DAILY**: 오늘 파티션만 추가.
- **EVENT**: `target_cve_ids`(보통 1건)만 MERGE — EPSS 값은 안 건드리고 KEV 필드만 최신화한다.
- **BACKFILL/INITIAL**: `target_cve_ids`(BACKFILL) 또는 전체(INITIAL)에 대해 과거 여러 날짜를 한 번에 만든다.

In [ ]:
%run 00_gold_common

### 1) 모드별 EPSS 소스 결정

In [ ]:
if run_mode == "DAILY":
    epss_source = spark.table(TBL["epss"]).filter(F.col("snapshot_date") == snapshot_date)
elif run_mode == "EVENT":
    if not target_cve_ids:
        raise ValueError("EVENT 모드는 target_cve_ids가 필요합니다.")
    base = spark.createDataFrame([(c, snapshot_date) for c in target_cve_ids], ["cve_id", "date_key"])
    epss_today = spark.table(TBL["epss"]).filter(
        (F.col("snapshot_date") == snapshot_date) & F.col("cve_id").isin(target_cve_ids)
    ).select("cve_id", F.col("snapshot_date").alias("date_key"), "epss_score", "epss_percentile")
    epss_source = base.join(epss_today, on=["cve_id", "date_key"], how="left")
elif run_mode in ("BACKFILL", "INITIAL"):
    epss_source = spark.table(TBL["epss"])
    if run_mode == "BACKFILL":
        if not target_cve_ids:
            raise ValueError("BACKFILL 모드는 target_cve_ids가 필요합니다.")
        epss_source = epss_source.filter(F.col("cve_id").isin(target_cve_ids))
else:
    raise ValueError(f"Unsupported run_mode: {run_mode}")

if "date_key" not in epss_source.columns:
    epss_source = epss_source.withColumnRenamed("snapshot_date", "date_key")

base_df = epss_source.select(
    "cve_id", "date_key", "epss_score", "epss_percentile"
).withColumn("has_epss", F.col("epss_score").isNotNull())


### 2) EPSS delta (전일/전주 대비)
같은 batch 안에 날짜가 여러 개면(BACKFILL/INITIAL) Window `lag`로, 단일 날짜(DAILY/EVENT)면 기존 `gold_vuln_daily`에서 -1일/-7일 값을 조회한다. 전일 파티션이 아예 없으면(최초 실행 등, 시나리오⑧) delta는 NULL로 남기고 건너뛴다 — 허위 EPSS_SPIKE 방지.

In [ ]:
from pyspark.sql import Window
from datetime import datetime as _dt, timedelta as _timedelta

def compute_deltas(df: DataFrame) -> DataFrame:
    distinct_dates = df.select("date_key").distinct().count()
    if distinct_dates > 1:
        w = Window.partitionBy("cve_id").orderBy("date_key")
        return (
            df
            .withColumn("_lag1", F.lag("epss_score", 1).over(w))
            .withColumn("_lag7", F.lag("epss_score", 7).over(w))
            .withColumn("epss_delta_1d", F.col("epss_score") - F.col("_lag1"))
            .withColumn("epss_delta_7d", F.col("epss_score") - F.col("_lag7"))
            .drop("_lag1", "_lag7")
        )

    if not table_exists(TBL["gold_vuln_daily"]):
        return (
            df.withColumn("epss_delta_1d", F.lit(None).cast(DoubleType()))
              .withColumn("epss_delta_7d", F.lit(None).cast(DoubleType()))
        )

    a_date_row = df.select("date_key").first()
    if a_date_row is None:
        return (
            df.withColumn("epss_delta_1d", F.lit(None).cast(DoubleType()))
              .withColumn("epss_delta_7d", F.lit(None).cast(DoubleType()))
        )

    d0 = _dt.strptime(a_date_row["date_key"], "%Y-%m-%d")
    d_minus1 = (d0 - _timedelta(days=1)).strftime("%Y-%m-%d")
    d_minus7 = (d0 - _timedelta(days=7)).strftime("%Y-%m-%d")

    hist = spark.table(TBL["gold_vuln_daily"]).filter(F.col("date_key").isin([d_minus1, d_minus7]))
    prev1 = hist.filter(F.col("date_key") == d_minus1).select("cve_id", F.col("epss_score").alias("_lag1"))
    prev7 = hist.filter(F.col("date_key") == d_minus7).select("cve_id", F.col("epss_score").alias("_lag7"))

    return (
        df.join(prev1, on="cve_id", how="left")
          .join(prev7, on="cve_id", how="left")
          .withColumn("epss_delta_1d", F.col("epss_score") - F.col("_lag1"))
          .withColumn("epss_delta_7d", F.col("epss_score") - F.col("_lag7"))
          .drop("_lag1", "_lag7")
    )

base_df = compute_deltas(base_df)


### 3) KEV as-of / GHSA 배열 / Systemic 플래그 붙이기

In [ ]:
daily_df = kev_asof_join(base_df, cve_col="cve_id", date_col="date_key")
daily_df = alias_ghsa_ids_join(daily_df, cve_col="cve_id")
daily_df = systemic_join(daily_df, date_col="date_key")

daily_df = (
    daily_df
    .withColumn("coverage_score", F.when(F.col("has_epss"), F.lit(1.0)).otherwise(F.lit(0.0)))
    .withColumn("source_batch_ids", F.array(F.lit(batch_id)))
    .withColumn("batch_id", F.lit(batch_id))
    .withColumn("ingested_at", F.lit(INGESTED_AT))
)
daily_count = daily_df.count()


### 4) MERGE (grain: cve_id × date_key, 재실행해도 멱등)

In [ ]:
if table_exists(TBL["gold_vuln_daily"]):
    target = DeltaTable.forName(spark, TBL["gold_vuln_daily"])
    (
        target.alias("t")
        .merge(daily_df.alias("s"), "t.cve_id = s.cve_id AND t.date_key = s.date_key")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    write_overwrite(daily_df, TBL["gold_vuln_daily"], partition_by=["date_key"])

log_batch_run("gold_vuln_daily_build", "SUCCESS", record_count=daily_count)
log_dq_metric("gold_vuln_daily_build", "rows_written", daily_count, passed=True, detail=f"run_mode={run_mode}")
print(f"[gold_vuln_daily_build] mode={run_mode} rows={daily_count} batch_id={batch_id}")
